# Worklet 9 — Placement Readiness & Career Intelligence Portal
## Lab 05: Governed MCP-style connector

**Team:** Vijay Kumar — 01FE23BCI083; Rishi Kulkarni — 01FE23BCI088; Prajwal Hugar — 01FE23BCI075; Rohit Yendegeri — 01FE23BCI001.

**Problem statement:** Help students understand resume/JD gaps and prepare with auditable, placement-officer-reviewed guidance.

| Item | This lab |
| --- | --- |
| Goal | Swap the evidence source while leaving the consuming agent unchanged. |
| Inputs | Identical synthetic profile stored in JSON, SQLite and an API-shaped local adapter. |
| Deliverable | Executable demonstration, assertions and recorded output |
| Acceptance | Source substitution yields identical output; invalid schema, wrong owner and revoked consent fail. |

Run top to bottom from this repository. The notebook imports `lab_support.py` and existing backend modules; it is independent of other notebooks, but is not a standalone Colab upload. All examples use synthetic data.


### 1. Environment and scope

Only local code and disposable fixtures are used. Dependencies: Python 3.11+ and the repository backend requirements. No secrets, real student records, or reference-notebook outputs are copied.


In [1]:
from pathlib import Path
import sys
folder=Path.cwd()
if not (folder/'lab_support.py').exists():folder=folder/'Assignments'
assert (folder/'lab_support.py').exists(), 'Run from the repository root or Assignments directory'
sys.path.insert(0,str(folder.resolve()))
from lab_support import *
print('Worklet 9: offline synthetic demonstration; no Gemini API calls')


Worklet 9: offline synthetic demonstration; no Gemini API calls


### 2. Worklet-specific implementation and experiment

The following code executes the stated scenario and deliberately exercises rejection paths where relevant.


In [2]:
import tempfile
with tempfile.TemporaryDirectory() as directory:
 path=Path(directory)/'profiles.json';path.write_text(json.dumps({'DEMO001':{'skills':['Python','SQL']}}))
 database=sqlite3.connect(':memory:');database.execute('CREATE TABLE profiles(id TEXT,data TEXT)')
 database.execute('INSERT INTO profiles VALUES(?,?)',('DEMO001',json.dumps({'skills':['Python','SQL']})))
 readers=[lambda student:json.loads(path.read_text())[student],lambda student:json.loads(database.execute('SELECT data FROM profiles WHERE id=?',(student,)).fetchone()[0]),lambda student:{'skills':['Python','SQL']}]
 agent=lambda gateway:coverage(gateway.call('DEMO001','DEMO001')['skills'],['Python','SQL','Git','Testing'])
 gateways=[Connector(reader) for reader in readers]
 assert [agent(g) for g in gateways]==[50,50,50]
 expect(PermissionError,lambda:gateways[0].call('DEMO002','DEMO001'))
 expect(PermissionError,lambda:gateways[0].call('DEMO001','DEMO001',False))
 expect(ValueError,lambda:Connector(lambda _:{}).call('DEMO001','DEMO001'))
 show({'same_agent_results':[agent(g) for g in gateways],'audit':gateways[0].audit})
 database.close()


Expected refusal: Owner and active consent required
Expected refusal: Owner and active consent required
Expected refusal: Invalid source schema
{
  "same_agent_results": [
    50,
    50,
    50
  ],
  "audit": [
    {
      "tool": "profile",
      "student": "DEMO001",
      "resultDigest": "7c589e61f6b31768e1f5e7da70788282418808253ec7e3fc106945b00bce4a55"
    },
    {
      "tool": "profile",
      "student": "DEMO001",
      "resultDigest": "7c589e61f6b31768e1f5e7da70788282418808253ec7e3fc106945b00bce4a55"
    }
  ]
}


### 3. Acceptance checks against the actual portal

The demonstration asserts its own contract above. This additional test calls the project implementation using disposable fixtures. Implementation references: `backend/tool_gateway.py`.


In [3]:
regression_result=regressions('test_runtime.RuntimeTests.test_mcp_schema_memory_scope_and_consent')
show(regression_result)


.
----------------------------------------------------------------------
Ran 1 test in 1.767s

OK

{
  "tests": 1,
  "failures": 0,
  "errors": 0
}


### 4. Results, reflection and limits

The API adapter here is simulated and clearly labelled. The regression exercises the real authenticated /api/mcp JSON-RPC endpoint. External MCP client interoperability is not asserted.

The output above is generated by execution, not an expected-output transcription. If any assertion or regression fails, this notebook is not accepted.

### 5. Deliverables and next step

- This executed notebook and its assertion output.
- Shared original teaching helpers in `lab_support.py`.
- Relevant production code and regression mapping above.
- Full sequence and provenance in `Assignments/README.md`.

Continue to Lab 6 to build on this contract.
